### Reading from bronze layer

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")


In [0]:
df.display()

### Data Transformations

In [0]:
## trimming the extra whitespace in all strings 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## change column names 
mapping = {
    'prd_id':'product_id',
    'prd_key':'product_key',
    'prd_nm':'product_name',
    'prd_cost':'product_cost',
    'prd_line':'product_line',
    'prd_start_dt':'product_start_date',
    'prd_end_dt':'product_end_date'
}
df1 = df.withColumnsRenamed(mapping)

df1 = df1.orderBy('product_id')
df1.display()

In [0]:
# Count number of rows in Spark DataFrame
row_count = df1.count()
print(f"Row count: {row_count}")

In [0]:
from pyspark.sql import functions as F

# Calculate Min, Max, Mean, and Median (50th percentile) per product line
stats_df = df1.groupBy("product_line").agg(
    F.min("product_cost").alias("min_cost"),
    F.max("product_cost").alias("max_cost"),
    F.mean("product_cost").alias("mean_cost"),
    F.percentile_approx("product_cost", 0.5).alias("median_cost"),
    F.count(F.when(F.col("product_cost").isNull(), 1)).alias("null_count")
)

stats_df.display()

In [0]:
## In product_cost column, replace all null values in the product_cost with a average of cost of all products in the same product_line
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define the window partitioned by product_line
window_spec = Window.partitionBy("product_line")

# Replace nulls in product_cost with the meidan of its corresponding product_line
df2 = df1.withColumn(
    "product_cost",
    F.when(
        F.col("product_cost").isNull(), 
        F.percentile_approx(F.col("product_cost"), 0.5).over(window_spec)
    ).otherwise(F.col("product_cost"))
)

df2 = df2.orderBy('product_id')
display(df2)

In [0]:
## split the product_key column into product_category_code and product_key
# Split by the second hyphen
# We use regex to capture everything before the 2nd hyphen and everything after it
df3 = df2.withColumn("product_category_code", F.regexp_extract("product_key", r"^(.*?-.*?)-", 1)) \
         .withColumn("product_key", F.regexp_extract("product_key", r"^(.*?-.*?)-(.*)$", 2))

display(df3)

In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df3.groupBy(df3.columns[0]).count().filter("count > 1")
display(duplicate_count)

### Write into silver table 


In [0]:
df3.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.crm_product_info_cleaned")